In [4]:
import os
import shutil
import numpy as np
from PIL import Image
import cv2
import yaml
base_path = r"D:\individual_task\MSFD"
yolo_path = r"D:\individual_task\MSFD_YOLO"  
folders = [
    'images/train',
    'images/val',
    'labels/train', 
    'labels/val',
    'binary_masks'  
]
for folder in folders:
    os.makedirs(os.path.join(yolo_path, folder), exist_ok=True)

In [5]:
def rgb_to_binary_mask(rgb_mask, threshold=10):
    gray = np.mean(rgb_mask, axis=2).astype(np.uint8)
    binary = (gray > threshold).astype(np.uint8) * 255
    return binary
def mask_to_yolo_format(binary_mask, class_id=0):
    contours, _ = cv2.findContours(
        binary_mask, 
        cv2.RETR_EXTERNAL,  
        cv2.CHAIN_APPROX_SIMPLE  
    )
    yolo_lines = []
    for contour in contours:
        epsilon = 0.002 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) < 3:
            continue
        height, width = binary_mask.shape
        normalized_coords = []
        for point in approx:
            x, y = point[0]
            norm_x = x / width
            norm_y = y / height
            normalized_coords.extend([norm_x, norm_y])
        yolo_line = f"{class_id} " + " ".join([f"{coord:.6f}" for coord in normalized_coords])
        yolo_lines.append(yolo_line)
    return yolo_lines
def process_single_pair(img_path, mask_path, output_img_path, output_label_path, output_mask_path=None):
    try:
        img = Image.open(img_path)
        mask_rgb = Image.open(mask_path)
        if img.size != mask_rgb.size:
            print(f"Размеры не совпадают: {img.size} & {mask_rgb.size}")
            mask_rgb = mask_rgb.resize(img.size, Image.Resampling.LANCZOS)
        img_array = np.array(img)
        mask_array = np.array(mask_rgb)
        img.save(output_img_path)
        binary_mask = rgb_to_binary_mask(mask_array, threshold=10)
        if output_mask_path:
            binary_img = Image.fromarray(binary_mask)
            binary_img.save(output_mask_path)
        yolo_lines = mask_to_yolo_format(binary_mask, class_id=0)
        if yolo_lines:
            with open(output_label_path, 'w') as f:
                f.write('\n'.join(yolo_lines))
            return True, len(yolo_lines)
        else:
            with open(output_label_path, 'w') as f:
                f.write('')
            return True, 0
    except Exception as e:
        print("Ошибка")
        return False, 0

In [7]:
import random
face_crop_path = os.path.join(base_path, "1", "face_crop")
segmentation_path = os.path.join(base_path, "1", "face_crop_segmentation")
image_files = [f for f in os.listdir(face_crop_path) if f.lower().endswith('.jpg')]
mask_files = [f for f in os.listdir(segmentation_path) if f.lower().endswith('.jpg')]
print(f"Всего изображений:{len(image_files)}")
print(f"Всего масок:{len(mask_files)}")
matching_pairs = []
missing_masks = []
for img_file in image_files:
    if img_file in mask_files:
        matching_pairs.append(img_file)
    else:
        missing_masks.append(img_file)
print(f"Найдено пар:{len(matching_pairs)}")
print(f"Изображений без масок:{len(missing_masks)}")
random.shuffle(matching_pairs)
split_idx = int(len(matching_pairs) * 0.8)
train_files = matching_pairs[:split_idx]
val_files = matching_pairs[split_idx:]
print(f"\nРазделение данных:")
print(f"Train:{len(train_files)}")
print(f"Val:{len(val_files)}")
stats = {
    'total': 0,
    'train': 0,
    'val': 0,
    'train_contours': 0,
    'val_contours': 0,
    'errors': 0
}
for i, filename in enumerate(train_files):
    if i % 1000 == 0:
        print(f"{i}/{len(train_files)}")
    img_path = os.path.join(face_crop_path, filename)
    mask_path = os.path.join(segmentation_path, filename)
    output_img_path = os.path.join(yolo_path, 'images/train', filename)
    output_label_path = os.path.join(yolo_path, 'labels/train', filename.replace('.jpg', '.txt'))
    output_mask_path = os.path.join(yolo_path, 'binary_masks', f"train_{filename}")
    success, num_contours = process_single_pair(
        img_path, mask_path, 
        output_img_path, output_label_path,
        output_mask_path
    )
    if success:
        stats['train'] += 1
        stats['train_contours'] += num_contours
    else:
        stats['errors'] += 1
for i, filename in enumerate(val_files):
    if i % 500 == 0:
        print(f"Обработано {i}/{len(val_files)}")
    img_path = os.path.join(face_crop_path, filename)
    mask_path = os.path.join(segmentation_path, filename)
    output_img_path = os.path.join(yolo_path, 'images/val', filename)
    output_label_path = os.path.join(yolo_path, 'labels/val', filename.replace('.jpg', '.txt'))
    output_mask_path = os.path.join(yolo_path, 'binary_masks', f"val_{filename}")
    success, num_contours = process_single_pair(
        img_path, mask_path, 
        output_img_path, output_label_path,
        output_mask_path
    )
    if success:
        stats['val'] += 1
        stats['val_contours'] += num_contours
    else:
        stats['errors'] += 1
stats['total'] = stats['train'] + stats['val']
print(f"Всего обработано:{stats['total']}")
print(f"Train:{stats['train']}")
print(f"Val:{stats['val']}")
print(f"Ошибок: {stats['errors']}")

Всего изображений:9383
Всего масок:9383
Найдено пар:9382
Изображений без масок:1

Разделение данных:
Train:7505
Val:1877
0/7505
1000/7505
2000/7505
3000/7505
Размеры не совпадают: (172, 129) & (172, 128)
4000/7505
5000/7505
6000/7505
7000/7505
Обработано 0/1877
Обработано 500/1877
Обработано 1000/1877
Обработано 1500/1877
Всего обработано:9382
Train:7505
Val:1877
Ошибок: 0


In [9]:
yaml_content = {
    'path': yolo_path,  
    'train': 'images/train', 
    'val': 'images/val',
    'nc': 1, 
    'names': ['face_mask'] 
}
yaml_path = os.path.join(yolo_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)
print("Содержимое data.yaml:")
print(yaml.dump(yaml_content, default_flow_style=False))

Содержимое data.yaml:
names:
- face_mask
nc: 1
path: D:\individual_task\MSFD_YOLO
train: images/train
val: images/val

